# 01 · LIBERO Plus 데이터·AVA/OFT 가중치 → Google Drive
CPU 런타임에서 시작할 수 있습니다. 저장소의 일반 코드 파일과 폴더를 Drive에 복사하고 공통 프로젝트 루트를 지정하십시오.
`src/scripts/configs/docs/experiments/third_party/ava_vla`와 루트 문서 구조를 유지하십시오. 소스 목록과 해시는 `experiments/manifests/colab_source.json`으로 검사합니다.
실행 순서: Drive 연결 → 환경 설치 → 공식 고정 revision 다운로드/해시 검증 → 분할 ZIP 결합/CRC 검사·해제 → episode 준비.
[데이터 원본](https://huggingface.co/datasets/Sylvest/libero_plus_rlds/tree/fb0c7029b076030d5d57227229e4f7460def1f7c)
은 **75.5 GB 분할 ZIP 한 개**입니다. `.z01/.z02/.zip`을 각각 해제하지 마십시오.
[AVA 초기 T0](https://huggingface.co/LiAuto-DSR/avavla-libero-4in1/tree/090bf68d234ea13bc78476236f82458b25b797b8)
는 backbone·tokenizer·processor·action head·proprio projector·attention generator 전체를 받습니다.
Drive에는 압축 원본+추출본+query 이미지+가중치를 둘 공간이, `/content`에는 결합 ZIP 약 75.5 GB가 필요합니다.
코드가 실제 파일 크기로 여유 공간을 검사합니다. Drive 계정 할당량은 Drive에서 별도로 확인하십시오.
오늘 로컬에서 다운로드/GPU 학습을 실행한 결과가 아닙니다. 각 셀을 실행하면 해당 다운로드·처리가 시작됩니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_MANIFEST_SHA256 = "caecbd305bd2be707702c4cdbc5b7b692cb5e6d94718944126d30569a9bd7423"
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'))
def run(args, env=None):
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('실행에 실패했습니다. 위 오류를 확인하십시오. 후속 학습은 시작하지 않습니다.')
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

In [ ]:
DATA_PY = environment('vlade_data_py311', '3.11', [
    'numpy==1.26.4', 'pillow==10.4.0', 'huggingface_hub==0.36.2',
    'tensorflow==2.15.0', 'tensorflow-datasets==4.9.3', 'tensorflow-metadata==1.15.0'])
if shutil.which('zip') is None:
    run(['apt-get', 'update'])
    run(['apt-get', 'install', '-y', 'zip'])

## 공식 파일 다운로드·검증
중단 후 같은 셀을 실행하면 Hub 다운로드 캐시를 재사용하고 파일 해시를 다시 검사합니다. 대용량 데이터를 로컬 컴퓨터로 거치지 않고 Drive에 저장합니다.

In [ ]:
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'data', '--root', ROOT])
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'ava', '--root', ROOT])
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'oft', '--root', ROOT])

In [ ]:
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'extract', '--root', ROOT,
     '--scratch', '/content/libero_zip_scratch'])

## 공통 episode 분할·정규화·관측 준비
확정 정본은 D/content-group 80/10/10, seed 42입니다. Train/Validation/Test는
11,477/1,428/1,442 Episode이고 Train 내부 Calibration은 728 Episode입니다.
준비 과정은 각 trajectory의 exact action+state group을 canonical manifest와 대조하며 임의 재분할을 금지합니다.
Train의 **전체 timestep**에서 exact q01/q99를 계산합니다. Validation/test는 통계에 넣지 않습니다.
Query 간격은 8이며 두 카메라 원본을 query 시점에만 보관합니다. 마지막 chunk의 padding은 loss에서 제외합니다.
이 prepared-data 경로는 augmentation 없이 FFT/BC/cache의 입력·정규화를 공유합니다.
중단된 `.incomplete`는 완료 데이터로 읽지 않습니다. 전처리 중단 시에는 새 output 경로로 다시 준비하십시오.

In [ ]:
PREPARED = ROOT / 'data/processed/libero_plus_episode80'
if (PREPARED / 'COMPLETE').is_file():
    print('준비된 데이터 재사용:', PREPARED)
else:
    run([DATA_PY, 'scripts/data/prepare_episode_data.py',
         '--tfds-root', ROOT / 'data/processed/libero_plus_tfds', '--output', PREPARED,
         '--canonical-manifest', CODE / 'experiments/manifests/libero_plus_rlds/canonical_d_seed42.jsonl.gz',
         '--canonical-sha256', '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80',
         '--canonical-normalization', CODE / 'experiments/manifests/libero_plus_rlds/normalization_train_d_seed42.json',
         '--canonical-normalization-sha256', 'eac95bcc81b6182b90a2bd030096df6e3e408cb88923d0a1d57dc6c81e297c67'])
manifest = json.loads((PREPARED / 'manifest.json').read_text())
print(json.dumps({key: manifest[key] for key in ['counts', 'calibration_episodes', 'split_policy',
                                               'canonical_manifest_sha256', 'canonical_normalization_sha256',
                                               'split_sha256', 'normalization_sha256', 'dataset_revision']}, indent=2))
print('다음: 02 Teacher 노트북을 A100 런타임에서 실행하십시오.')